# KAS-PPI: thực nghiệm cơ bản (06/10/2026)

Chạy lại toàn bộ kết quả trong tài liệu *Chốt mô hình luận văn: KAS-PPI*:
1. Tái hiện đặc trưng KGsim2vec (Python) và biểu diễn SEEK (kiểm tra khớp với file của tác giả).
2. 10-fold cho 3 view (KGsim2vec, SEEK, Hybrid) × RF/XGB, có và không có aspect `protein-containing complex`.
3. Giải thích bằng KAS và các baseline (LOO, FS0, TreeSHAP, LIME, ngẫu nhiên), đo các tiêu chí VXAI.

Chỉ cần CPU. Bước 3 đầy đủ (18 lượt, 4.800 cặp) mất khoảng 1,5–2 giờ trên Colab; ô cấu hình bên dưới có chế độ nhanh.

In [ ]:
!git clone -q --depth 1 https://github.com/liseda-lab/ExplainablePPI.git
!git clone -q --depth 1 https://github.com/liseda-lab/seek.git
!mkdir -p godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/go-basic.zip -d godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/goa_human.zip -d godata
!cd seek/PPI && unzip -o -q go.zip && unzip -o -q go_annotations.zip && unzip -o -q Embeddings/Emb_classes_maxdepth4_nwalks100.zip -d Embeddings && unzip -o -q Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.zip -d Embeddings
!pip -q install shap xgboost

## Mã nguồn
Mỗi ô ghi một file. `kgsim.py`: đặc trưng KGsim2vec (Resnik-max, IC Seco trong từng aspect). `seekrep.py`: biểu diễn SEEK (trung bình embedding RDF2Vec của các DCA, embedding lớp cố định của tác giả). `explain_v2.py`: bộ giải thích KAS và các baseline.

In [ ]:
%%writefile kgsim.py
"""Python re-implementation of KGsim2vec features (Resnik-max, IC Seco computed inside each aspect subgraph)."""
import re, math, collections, pickle, os
OBO="http://purl.obolibrary.org/obo/GO_"
def parse_go(owl):
    parents=collections.defaultdict(set); label={}; cur=None
    pat_cls=re.compile(r'<owl:Class rdf:about="http://purl.obolibrary.org/obo/(GO_\d+)">')
    pat_sub=re.compile(r'<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/(GO_\d+)"/>')
    pat_lab=re.compile(r'<rdfs:label[^>]*>([^<]*)</rdfs:label>')
    depth=0
    for line in open(owl,encoding="utf-8"):
        m=pat_cls.search(line)
        if m: cur=m.group(1); continue
        if cur is None: continue
        if "</owl:Class>" in line and line.strip()=="</owl:Class>": cur=None; continue
        m=pat_sub.search(line)
        if m: parents[cur].add(m.group(1)); continue
        m=pat_lab.search(line)
        if m and cur not in label: label[cur]=m.group(1)
    return parents,label
def load_gaf(gaf):
    ann=collections.defaultdict(set)
    for l in open(gaf):
        if l.startswith("!"): continue
        f=l.split("\t")
        if len(f)<5 or "NOT" in f[3]: continue
        ann[f[1]].add("GO_"+f[4].split(":")[1])
    return ann
class KG:
    def __init__(s,owl,gaf,aspects):
        s.parents,s.label=parse_go(owl)
        s.children=collections.defaultdict(set)
        for c,ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        s.ann=load_gaf(gaf)
        s.aspects=aspects
        s._anc={}
        s.sub=[]; s.ic=[]
        for a in aspects:
            members=s.desc(a)|{a}
            nprot=sum(1 for p,ts in s.ann.items() if ts & members)
            N=len(members)+nprot   # SML counts annotated instances as vertices of the reduced graph
            ic={}
            for c in members:
                d=len(s.desc(c))
                ic[c]=1-math.log(d+1)/math.log(N) if N>1 else 1.0
            s.sub.append(members); s.ic.append(ic)
    def desc(s,c,_cache={}):
        if c in _cache: return _cache[c]
        out=set(); st=[c]
        while st:
            x=st.pop()
            for y in s.children.get(x,()):
                if y not in out: out.add(y); st.append(y)
        _cache[c]=out; return out
    def anc(s,c):
        if c in s._anc: return s._anc[c]
        out={c}; st=[c]
        while st:
            x=st.pop()
            for y in s.parents.get(x,()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c]=out; return out
    def closure(s,terms):
        out=set()
        for t in terms: out|=s.anc(t)
        return out
    def features(s,t1,t2):
        """t1,t2: sets of direct annotation terms."""
        A1=s.closure(t1); A2=s.closure(t2)
        common=A1&A2
        v=[]
        for i,a in enumerate(s.aspects):
            m=s.sub[i]
            # protein needs >=1 direct annotation in aspect
            if not (t1 & m) or not (t2 & m): v.append(0.0); continue
            cm=common & m
            v.append(max((s.ic[i][c] for c in cm),default=0.0))
        return v


Writing kgsim.py


In [ ]:
%%writefile seekrep.py
"""Python re-implementation of SEEK pair representations (Sousa et al., KR 2023):
pair vector = average RDF2Vec embedding of the disjoint common ancestors (DCA) of the two proteins' GO annotations.
Class embeddings are the authors' precomputed ones (fixed); only the DCA set changes when annotations are removed."""
import re, json, collections, numpy as np
from kgsim import parse_go
class SEEK:
    def __init__(s, owl, gaf, emb_file):
        s.parents, s.label = parse_go(owl)
        s.children = collections.defaultdict(set)
        for c, ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        # SEEK's build_kg reads every GAF line (qualifier column ignored), protein id = column 2
        s.ann = collections.defaultdict(set)
        with open(gaf) as f:
            f.readline()
            for l in f:
                if l.startswith("!"): continue
                c = l.split("\t")
                s.ann[c[1]].add("GO_" + c[4].split(":")[1])
        txt = open(emb_file).read().replace("'", '"')
        raw = json.loads(re.sub(r'"http://purl.obolibrary.org/obo/', '"', txt))
        s.emb = {k: np.asarray(v, dtype=np.float64) for k, v in raw.items()}
        s.dim = len(next(iter(s.emb.values())))
        s._anc = {}
    def anc(s, c):
        if c in s._anc: return s._anc[c]
        out = {c}; st = [c]
        while st:
            x = st.pop()
            for y in s.parents.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c] = frozenset(out); return s._anc[c]
    def desc(s, c, _cache={}):
        if c in _cache: return _cache[c]
        out = set(); st = [c]
        while st:
            x = st.pop()
            for y in s.children.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        _cache[c] = out; return out
    def dca(s, t1, t2):
        A1 = set().union(*[s.anc(t) for t in t1]) if t1 else set()
        A2 = set().union(*[s.anc(t) for t in t2]) if t2 else set()
        common = A1 & A2
        strict = set()
        for d in common: strict |= (s.anc(d) - {d})
        return [c for c in common - strict if c in s.emb]
    def rep(s, t1, t2):
        d = s.dca(t1, t2)
        if not d: return np.zeros(s.dim)
        return np.mean([s.emb[c] for c in d], 0)


Writing seekrep.py


In [ ]:
%%writefile build_feats.py
import sys,time,numpy as np,pickle
sys.path.insert(0,".")
from kgsim import KG
R="ExplainablePPI/Experimental_Results/"
aspects=[l.split("\t")[0].split("/")[-1] for l in open(R+"Results/kgsim2vec_sas.txt") if l.strip()]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects)
rows=[l.rstrip("\n").split("\t") for l in open(R+"Results/ss_resnik_max_ICseco.txt")]
lab={}
for l in open(R+"Data/v11(score950).txt"):
    f=l.split(); lab[(f[0],f[1])]=int(f[2])
keys=[(r[0].replace("http://",""),r[1].replace("http://","")) for r in rows]
Xref=np.array([[float(x) for x in r[2:]] for r in rows])
t=time.time()
Xpy=np.array([kg.features(kg.ann[a],kg.ann[b]) for a,b in keys])
print("built",time.time()-t)
y=np.array([lab[k] for k in keys])
np.savez("feats.npz",Xref=Xref,Xpy=Xpy,y=y,keys=np.array(keys),aspects=np.array(aspects))
d=np.abs(Xref-Xpy)
print("exact pairs",(d.max(1)<1e-6).mean(),"cells within 1e-6",(d<1e-6).mean(),"within 0.05",(d<0.05).mean())
print("zero-pattern agreement",((Xref>0)==(Xpy>0)).mean())
cors=[np.corrcoef(Xref[:,j],Xpy[:,j])[0,1] for j in range(52) if Xref[:,j].std()>0 and Xpy[:,j].std()>0]
print("per-aspect pearson median %.4f min %.4f"%(np.median(cors),np.min(cors)))


Writing build_feats.py


In [ ]:
%%writefile check_seek.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
t=time.time(); sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt"); print("load",time.time()-t, len(sk.emb))
ref={}
for l in open(S+"Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.txt"):
    f=l.rstrip("\n").split("\t"); ref[(f[0],f[1])]=np.array(f[2:],float)
pairs=[l.split("\t")[:2] for l in open(S+"STRING_pairs.txt")]
rng=np.random.default_rng(0); idx=rng.choice(len(pairs),1000,replace=False)
err=[];t=time.time()
for i in idx:
    a,b=pairs[i]; v=sk.rep(sk.ann[a],sk.ann[b]); err.append(np.abs(v-ref[(a,b)]).max())
err=np.array(err); print("time/pair",(time.time()-t)/1000, "exact(1e-6)",(err<1e-6).mean(), "max",err.max(), "p99",np.quantile(err,.99))


Writing check_seek.py


In [ ]:
%%writefile build_seek_feats.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
K=np.load("feats_kg.npz"); aspects=list(K["aspects"]); keys=[tuple(k) for k in K["keys"]]
roots=["GO_0008150","GO_0003674","GO_0005575"]
ch=set().union(*[sk.children[r] for r in roots])
print("aspects in SEEK GO:",len(ch),"overlap with KG aspects",len(ch&set(aspects)),"missing",set(aspects)-ch)
cx=sk.desc("GO_0032991")|{"GO_0032991"}
Xs=np.zeros((len(keys),sk.dim)); Xs51=np.zeros_like(Xs); t=time.time()
for i,(a,b) in enumerate(keys):
    t1,t2=sk.ann[a],sk.ann[b]
    Xs[i]=sk.rep(t1,t2); Xs51[i]=sk.rep(t1-cx,t2-cx)
print("built",time.time()-t, "rows changed by complex removal",(np.abs(Xs-Xs51).max(1)>0).mean())
np.savez("feats_seek.npz",Xseek=Xs,Xseek51=Xs51)


Writing build_seek_feats.py


In [ ]:
%%writefile train_v2.py
"""10-fold prediction table for the three pipelines (repo partitions), with and without the circular aspect."""
import numpy as np, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from scipy.stats import wilcoxon
K=np.load("feats_kg.npz"); S=np.load("feats_seek.npz"); y=K["y"]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]
P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds=[[int(l) for l in open(P+("1(complete)" if r==1 else str(r))+".txt") if l.strip()] for r in range(1,11)]
feats={"KG_52":K["Xpy"],"KG_51":K["Xpy"][:,keep],"SEEK_52":S["Xseek"],"SEEK_51":S["Xseek51"],
       "HYB_52":np.hstack([K["Xpy"],S["Xseek"]]),"HYB_51":np.hstack([K["Xpy"][:,keep],S["Xseek51"]])}
out={}
for mname in ["RF100","XGB300"]:
    for fname,X in feats.items():
        f1s=[];aucs=[]
        for test in folds:
            m=np.zeros(len(y),bool); m[test]=True
            clf=RandomForestClassifier(100,n_jobs=-1,random_state=0) if mname=="RF100" else XGBClassifier(n_estimators=300,n_jobs=-1,random_state=0,eval_metric="logloss")
            clf.fit(X[~m],y[~m]); p=clf.predict_proba(X[m])[:,1]
            f1s.append(f1_score(y[m],(p>0.5).astype(int),average="weighted")); aucs.append(roc_auc_score(y[m],p))
        out[f"{fname}|{mname}"]=dict(median_waf=float(np.median(f1s)),iqr=float(np.percentile(f1s,75)-np.percentile(f1s,25)),median_auc=float(np.median(aucs)),folds=[round(x,4) for x in f1s],aucs=[round(x,4) for x in aucs])
        print(fname,mname,"WAF %.4f IQR %.4f AUC %.4f"%(np.median(f1s),out[f"{fname}|{mname}"]["iqr"],np.median(aucs)),flush=True)
    for a,b in [("HYB_51","KG_51"),("KG_51","SEEK_51"),("HYB_51","SEEK_51"),("SEEK_52","SEEK_51")]:
        fa=np.array(out[f"{a}|{mname}"]["folds"]); fb=np.array(out[f"{b}|{mname}"]["folds"])
        out[f"wilcoxon|{mname}|{a}_vs_{b}"]=dict(mean_diff=float((fa-fb).mean()),wins=int((fa>fb).sum()),p=float(wilcoxon(fa,fb).pvalue))
        print(mname,a,"vs",b,out[f"wilcoxon|{mname}|{a}_vs_{b}"],flush=True)
json.dump(out,open("results_prediction_v2.json","w"),indent=1)


Writing train_v2.py


In [ ]:
%%writefile explain_v2.py
"""Direction D, full basic experiment: knowledge-space aspect Shapley (D) on three pipelines.
usage: python3 explain_v2.py PIPE MODEL FOLD N    PIPE in {kg,seek,hyb}, MODEL in {rf,xgb}
Players = GO aspects (51 direct children of the 3 GO roots, 'protein-containing complex' removed) in which
both proteins have >=1 direct annotation. Removing a player = deleting both proteins' annotations under that
aspect and recomputing the features through the pipeline (KGsim2vec: Resnik-max/IC-Seco; SEEK: DCA + mean
RDF2Vec embedding with the authors' fixed class embeddings)."""
import sys, time, json, itertools, math, random
import numpy as np
sys.path.insert(0,"."); sys.path.insert(0,"ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap
from lime import lime_tabular

PIPE,MODEL,FOLD,N_PAIRS=sys.argv[1],sys.argv[2],int(sys.argv[3]),int(sys.argv[4])
M_PERM=100; EXACT_MAX=8; TOPK=5
rng=np.random.default_rng(FOLD); random.seed(FOLD)

K=np.load("feats_kg.npz"); y=K["y"]; keys=[tuple(k) for k in K["keys"]]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]; A=[aspects[j] for j in keep]
SF=np.load("feats_seek.npz")
X={"kg":K["Xpy"][:,keep],"seek":SF["Xseek51"],"hyb":np.hstack([K["Xpy"][:,keep],SF["Xseek51"]])}[PIPE]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects); sub_kg=[kg.sub[j] for j in keep]
S_="seek/PPI/"
sk=SEEK(S_+"go.owl",S_+"go_annotations.gaf",S_+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
CX=sk.desc("GO_0032991")|{"GO_0032991"}
sub_sk=[sk.desc(a)|{a} for a in A]

P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
test=[int(l) for l in open(P+("1(complete)" if FOLD==1 else str(FOLD))+".txt") if l.strip()]
m=np.zeros(len(y),bool); m[test]=True
clf=(RandomForestClassifier(100,n_jobs=1,random_state=0) if MODEL=="rf"
     else XGBClassifier(n_estimators=300,n_jobs=1,random_state=0,eval_metric="logloss")).fit(X[~m],y[~m])
def proba(Z): return clf.predict_proba(np.atleast_2d(Z))[:,1].astype(np.float64)

# state = (kg annotations p1, kg annotations p2, seek annotations p1, seek annotations p2)
def feats(st):
    k1,k2,s1,s2=st; parts=[]
    if PIPE in("kg","hyb"):
        v=kg.features(k1,k2); parts.append([v[j] for j in keep])
    if PIPE in("seek","hyb"): parts.append(list(sk.rep(s1,s2)))
    return np.concatenate(parts)
def players_of(st):
    k1,k2,s1,s2=st; out=[]
    for i in range(len(A)):
        a=(PIPE!="seek") and bool(k1&sub_kg[i]) and bool(k2&sub_kg[i])
        b=(PIPE!="kg") and bool(s1&sub_sk[i]) and bool(s2&sub_sk[i])
        if a or b: out.append(i)
    return out
def remove(st,S):
    if not S: return st
    rk=set().union(*[sub_kg[i] for i in S]); rs=set().union(*[sub_sk[i] for i in S])
    k1,k2,s1,s2=st; return (k1-rk,k2-rk,s1-rs,s2-rs)

def shapley(value_batch, players, seed):
    k=len(players); phi=np.zeros(len(A))
    if k==0: return phi
    if k<=EXACT_MAX:
        subsets=[frozenset(c) for r in range(k+1) for c in itertools.combinations(players,r)]
        vals=dict(zip(subsets,value_batch(subsets)))
        for i in players:
            others=[p for p in players if p!=i]
            for r in range(k):
                w=math.factorial(r)*math.factorial(k-r-1)/math.factorial(k)
                for c in itertools.combinations(others,r):
                    S=frozenset(c); phi[i]+=w*(vals[S]-vals[S|{i}])
        return phi
    r=random.Random(seed); perms=[r.sample(players,k) for _ in range(M_PERM)]
    subsets=set()
    for p in perms:
        for t in range(k+1): subsets.add(frozenset(p[:t]))
    subsets=list(subsets); vals=dict(zip(subsets,value_batch(subsets)))
    for p in perms:
        for t,i in enumerate(p): phi[i]+=vals[frozenset(p[:t])]-vals[frozenset(p[:t+1])]
    return phi/M_PERM

def explain_D(st,seed=0):
    pl=players_of(st)
    return shapley(lambda subs: proba(np.array([feats(remove(st,S)) for S in subs])),pl,seed)
def explain_LOO(st,seed=0):   # leave-one-aspect-out occlusion in knowledge space (aspect-level analogue of SEEK 'necessary')
    pl=players_of(st); phi=np.zeros(len(A))
    if not pl: return phi
    v=proba(np.array([feats(st)]+[feats(remove(st,[i])) for i in pl]))
    for j,i in enumerate(pl): phi[i]=v[0]-v[j+1]
    return phi
def explain_FS0(st,seed=0):   # KGsim2vec only: same players and sampler, feature set to 0 instead of recomputed
    pl=players_of(st); x=feats(st)
    def vb(subs):
        Z=np.repeat(x[None],len(subs),0)
        for r,S in enumerate(subs): Z[r,list(S)]=0
        return proba(Z)
    return shapley(vb,pl,seed)
# bg=X[~m][rng.choice((~m).sum(),100,replace=False)]
# tree=shap.TreeExplainer(clf,data=bg,feature_perturbation="interventional")
if MODEL == "xgb":
    tree = shap.TreeExplainer(clf, feature_perturbation="tree_path_dependent")
else:
    bg = X[~m][rng.choice((~m).sum(), 100, replace=False)]
    tree = shap.TreeExplainer(clf, data=bg, feature_perturbation="interventional")
def explain_TS(st,seed=0):    # kg: per aspect feature; hyb: only the 51 named KGsim2vec columns map to aspects
    x=feats(st); sv=tree.shap_values(x[None],check_additivity=False)
    sv=sv[1] if isinstance(sv,list) else sv
    sv=np.array(sv); sv=sv[0,:,1] if sv.ndim==3 else sv[0]
    return np.asarray(sv[:len(A)],dtype=np.float64)
if PIPE=="kg":
    lime_exp=lime_tabular.LimeTabularExplainer(X[~m][:2000],feature_names=A,class_names=["0","1"],discretize_continuous=True,random_state=0)
def explain_LIME(st,seed=0):
    x=feats(st); lime_exp.random_state=np.random.RandomState(seed)
    e=lime_exp.explain_instance(x,lambda Z: clf.predict_proba(Z).astype(np.float64),num_features=10,num_samples=2000,labels=(1,))
    phi=np.zeros(len(A))
    for j,w in e.as_map()[1]: phi[j]=w
    return phi
METHODS={"D":explain_D,"LOO":explain_LOO}
if PIPE=="kg": METHODS.update({"FS0":explain_FS0,"TreeSHAP":explain_TS,"LIME":explain_LIME})
if PIPE=="hyb": METHODS.update({"TreeSHAP_KGpart":explain_TS})
DETERMINISTIC={"LOO","TreeSHAP","TreeSHAP_KGpart"}

def ranking(phi,pred):
    s=phi if pred==1 else -phi
    return [int(j) for j in np.argsort(-s) if s[j]>0]
def deletion_ks(st,order,pred):
    Z=np.array([feats(remove(st,order[:k])) for k in range(1,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def insertion_ks(st,order,pred,pl):  # start with every player removed, add back the top-k
    Z=np.array([feats(remove(st,[i for i in pl if i not in order[:k]])) for k in range(0,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def eff_size(phi):
    a=np.sort(np.abs(phi))[::-1]; tot=a.sum()
    return int(np.searchsorted(np.cumsum(a)/tot,0.8)+1) if tot>0 else 0
def jacc(a,b,k=3):
    A_,B_=set(a[:k]),set(b[:k]); return len(A_&B_)/len(A_|B_) if (A_|B_) else 1.0

pos=[i for i in test if y[i]==1]; neg=[i for i in test if y[i]==0]
idx=list(rng.choice(pos,N_PAIRS//2,replace=False))+list(rng.choice(neg,N_PAIRS//2,replace=False))
rec=[]; t0=time.time()
for n,i in enumerate(idx):
    p1,p2=keys[i]
    st=(set(kg.ann[p1]),set(kg.ann[p2]),set(sk.ann[p1])-CX,set(sk.ann[p2])-CX)
    p=proba(feats(st))[0]; pred=int(p>0.5); conf=p if pred else 1-p
    pl=players_of(st)
    r=dict(i=int(i),pair=f"{p1}-{p2}",y=int(y[i]),pred=pred,conf=float(conf),k=len(pl))
    # continuity: drop ONE random direct annotation (same GO term in both annotation sets) lying in a player aspect
    cand=sorted({(w,t) for w in (0,1) for t in (st[w]|st[w+2]) if any(t in sub_kg[a] or t in sub_sk[a] for a in pl)})
    pert=None
    if cand:
        w,t=cand[rng.integers(len(cand))]; q=list(st); q[w]=q[w]-{t}; q[w+2]=q[w+2]-{t}; q=tuple(q)
        if int(proba(feats(q))[0]>0.5)==pred: pert=q
    ro=[int(v) for v in rng.permutation(pl)] if pl else []
    r["del_random"]=deletion_ks(st,ro,pred) if ro else None
    r["ins_random"]=insertion_ks(st,ro,pred,pl) if ro else None
    for name,f in METHODS.items():
        ts_=time.time(); phi=f(st,0); r[f"time_{name}"]=time.time()-ts_
        order=ranking(phi,pred)
        r[f"cov_{name}"]=int(len(order)>0); r[f"size_{name}"]=eff_size(phi)
        r[f"del_{name}"]=deletion_ks(st,order,pred) if order else None
        r[f"ins_{name}"]=insertion_ks(st,order,pred,pl) if order else None
        r[f"top3_{name}"]=[A[j] for j in order[:3]]
        r[f"phi_{name}"]={A[j]:round(float(phi[j]),5) for j in np.nonzero(phi)[0]}
        if name in DETERMINISTIC or (name=="D" and len(pl)<=EXACT_MAX):
            r[f"cons_jacc_{name}"]=1.0; r[f"cons_rho_{name}"]=1.0
        else:
            phi2=f(st,1); o2=ranking(phi2,pred); r[f"cons_jacc_{name}"]=jacc(order,o2)
            r[f"cons_rho_{name}"]=float(spearmanr(phi,phi2).correlation) if np.std(phi)>0 and np.std(phi2)>0 else None
        r[f"cont_{name}"]=jacc(order,ranking(f(pert,0),pred)) if pert is not None else None
    rec.append(r)
    if n%25==0: print(PIPE,MODEL,FOLD,n,round(time.time()-t0,1),"s k=",len(pl),flush=True)
json.dump(dict(pipe=PIPE,model=MODEL,fold=FOLD,aspects=A,labels={a:kg.label.get(a,a) for a in A},records=rec),open(f"rec_{PIPE}_{MODEL}_f{FOLD}_{N_PAIRS}.json","w"))
print("done",time.time()-t0)


Overwriting explain_v2.py


In [ ]:
%%writefile analyze_v2.py
"""Summaries for explain_v2 runs: per run and pooled per (pipeline, model) over folds."""
import json, glob, collections, numpy as np
from scipy.stats import wilcoxon
rng=np.random.default_rng(0)
def curve(r,key,lo=0):
    v=r.get(key); return float(np.mean(v[lo:])) if v else None
def ci(a):
    b=[np.mean(rng.choice(a,len(a))) for _ in range(2000)]; return [float(x) for x in np.percentile(b,[2.5,97.5])]
def summarize(R,methods):
    out={"n":len(R),"acc":float(np.mean([r["pred"]==r["y"] for r in R])),
         "k_median":float(np.median([r["k"] for r in R])),"k_zero":int(sum(r["k"]==0 for r in R))}
    dels={};inss={}
    for m in methods+["random"]:
        # empty explanation: nothing is removed (deletion stays at conf), nothing is added back (insertion stays at base)
        dels[m]=np.array([curve(r,f"del_{m}") if r.get(f"del_{m}") else r["conf"] for r in R])
        inss[m]=np.array([curve(r,f"ins_{m}",1) if r.get(f"ins_{m}") else (r["ins_random"][0] if r.get("ins_random") else r["conf"]) for r in R])
        out[f"del_{m}"]=dict(mean=float(dels[m].mean()),ci95=ci(dels[m]))
        out[f"ins_{m}"]=dict(mean=float(inss[m].mean()),ci95=ci(inss[m]))
    for m in methods[1:]+["random"]:
        for nm,arr,sign in [("del",dels,1),("ins",inss,-1)]:
            a,b=arr["D"],arr[m]; d=(a-b)*sign
            try: p=float(wilcoxon(a,b).pvalue)
            except ValueError: p=1.0
            out[f"wil_{nm}_D_vs_{m}"]=dict(diff=float((a-b).mean()),D_better=float((d<0).mean()),ties=float((d==0).mean()),p=p)
    for m in methods:
        out[f"pars_{m}"]=float(np.mean([r[f"size_{m}"] for r in R]))
        out[f"cov_{m}"]=float(np.mean([r[f"cov_{m}"] for r in R]))
        out[f"cons_{m}"]=float(np.mean([r[f"cons_jacc_{m}"] for r in R]))
        c=[r[f"cont_{m}"] for r in R if r[f"cont_{m}"] is not None]; out[f"cont_{m}"]=float(np.mean(c)); out["cont_n"]=len(c)
        out[f"time_{m}"]=float(np.median([r[f"time_{m}"] for r in R]))
    if "LOO" in methods: out["D_vs_LOO_top3_differs"]=float(np.mean([r["top3_D"]!=r["top3_LOO"] for r in R]))
    if "FS0" in methods: out["D_vs_FS0_top3_differs"]=float(np.mean([r["top3_D"]!=r["top3_FS0"] for r in R]))
    return out
runs={}; pooled=collections.defaultdict(list); labels={}
for f in sorted(glob.glob("rec_*_f*_*.json")):
    J=json.load(open(f)); labels=J["labels"]; R=J["records"]
    methods=[k[5:] for k in R[0] if k.startswith("time_")]
    runs[f]=summarize(R,methods); pooled[(J["pipe"],J["model"])]+=R
allout={"runs":runs,"pooled":{}}
for (p,mo),R in pooled.items():
    methods=[k[5:] for k in R[0] if k.startswith("time_")]
    s=summarize(R,methods); allout["pooled"][f"{p}|{mo}"]=s
    # top aspects in D explanations of correctly predicted positives (input for a plausibility case study)
    cnt=collections.Counter(a for r in R if r["y"]==1 and r["pred"]==1 for a in r["top3_D"][:1])
    s["top1_aspect_TP"]=[(labels.get(a,a),c) for a,c in cnt.most_common(8)]
json.dump(allout,open("summary_v2.json","w"),indent=1)
for key,s in allout["pooled"].items():
    print("=====",key,"n",s["n"],"acc %.3f"%s["acc"],"k_med",s["k_median"])
    ms=[k[4:] for k in s if k.startswith("del_")]
    for m in ms: print("  %-16s del %.3f  ins %.3f"%(m,s[f"del_{m}"]["mean"],s[f"ins_{m}"]["mean"]),
                       "" if m=="random" else "pars %.2f cov %.3f cons %.3f cont %.3f t %.3f"%(s[f"pars_{m}"],s[f"cov_{m}"],s[f"cons_{m}"],s[f"cont_{m}"],s[f"time_{m}"]))
    for k,v in s.items():
        if k.startswith("wil_"): print("  ",k,{a:(round(b,4) if isinstance(b,float) else b) for a,b in v.items()})


Writing analyze_v2.py


## 1. Đặc trưng và kiểm tra tái hiện

In [ ]:
!python build_feats.py && cp feats.npz feats_kg.npz
!python check_seek.py
!python build_seek_feats.py

built 22.98564124107361
exact pairs 0.03497942386831276 cells within 1e-6 0.9616685921096001 within 0.05 0.9989548761026424
zero-pattern agreement 0.9993110014587678
per-aspect pearson median 1.0000 min 0.7884
load 12.980549335479736 50422
time/pair 0.00040970587730407716 exact(1e-6) 1.0 max 0.0 p99 0.0
aspects in SEEK GO: 52 overlap with KG aspects 52 missing set()
built 15.682819366455078 rows changed by complex removal 0.4109711085656103


## 2. Dự đoán 10-fold (3 view × RF/XGB, 51 và 52 aspect)

In [ ]:
!python train_v2.py

KG_52 RF100 WAF 0.9306 IQR 0.0047 AUC 0.9780
KG_51 RF100 WAF 0.9246 IQR 0.0043 AUC 0.9746
SEEK_52 RF100 WAF 0.9073 IQR 0.0077 AUC 0.9668
SEEK_51 RF100 WAF 0.9039 IQR 0.0056 AUC 0.9646
HYB_52 RF100 WAF 0.9283 IQR 0.0056 AUC 0.9770
HYB_51 RF100 WAF 0.9247 IQR 0.0039 AUC 0.9753
RF100 HYB_51 vs KG_51 {'mean_diff': -0.0009100000000000108, 'wins': 4, 'p': 0.556640625}
RF100 KG_51 vs SEEK_51 {'mean_diff': 0.02128000000000002, 'wins': 10, 'p': 0.001953125}
RF100 HYB_51 vs SEEK_51 {'mean_diff': 0.02037000000000001, 'wins': 10, 'p': 0.001953125}
RF100 SEEK_52 vs SEEK_51 {'mean_diff': 0.004289999999999983, 'wins': 10, 'p': 0.001953125}
KG_52 XGB300 WAF 0.9315 IQR 0.0031 AUC 0.9792
KG_51 XGB300 WAF 0.9253 IQR 0.0032 AUC 0.9764
SEEK_52 XGB300 WAF 0.9161 IQR 0.0060 AUC 0.9717
SEEK_51 XGB300 WAF 0.9104 IQR 0.0072 AUC 0.9689
HYB_52 XGB300 WAF 0.9350 IQR 0.0031 AUC 0.9801
HYB_51 XGB300 WAF 0.9286 IQR 0.0052 AUC 0.9779
XGB300 HYB_51 vs KG_51 {'mean_diff': 0.0031200000000000338, 'wins': 8, 'p': 0.0097656

## 3. Giải thích và tiêu chí VXAI
`FULL = True` chạy đúng cấu hình trong tài liệu (fold 1: 400 cặp; fold 2–3: 200 cặp). `FULL = False` chỉ chạy fold 1 với 100 cặp mỗi cấu hình (khoảng 15 phút).

In [ ]:
FULL = True
import subprocess
jobs = [(p, m, 1, 400 if FULL else 100) for p in ['kg','seek','hyb'] for m in ['rf','xgb']]
if FULL: jobs += [(p, m, f, 200) for f in [2, 3] for p in ['kg','seek','hyb'] for m in ['rf','xgb']]
for p, m, f, n in jobs:
    print(p, m, f, n, flush=True)
    subprocess.run(['python', 'explain_v2.py', p, m, str(f), str(n)], check=True)

kg rf 1 400
kg xgb 1 400
seek rf 1 400
seek xgb 1 400
hyb rf 1 400
hyb xgb 1 400
kg rf 2 200
kg xgb 2 200
seek rf 2 200
seek xgb 2 200
hyb rf 2 200
hyb xgb 2 200
kg rf 3 200
kg xgb 3 200
seek rf 3 200
seek xgb 3 200
hyb rf 3 200
hyb xgb 3 200


In [ ]:
!python analyze_v2.py

===== hyb|rf n 800 acc 0.943 k_med 5.0
  D                del 0.483  ins 0.846 pars 3.17 cov 0.986 cons 0.990 cont 0.952 t 0.083
  LOO              del 0.554  ins 0.821 pars 2.68 cov 0.975 cons 1.000 cont 0.927 t 0.014
  TreeSHAP_KGpart  del 0.587  ins 0.795 pars 2.96 cov 1.000 cons 1.000 cont 0.971 t 0.270
  random           del 0.641  ins 0.714 
   wil_del_D_vs_LOO {'diff': -0.0709, 'D_better': 0.6388, 'ties': 0.1613, 'p': 0.0}
   wil_ins_D_vs_LOO {'diff': 0.025, 'D_better': 0.4838, 'ties': 0.165, 'p': 0.0}
   wil_del_D_vs_TreeSHAP_KGpart {'diff': -0.1039, 'D_better': 0.8, 'ties': 0.0488, 'p': 0.0}
   wil_ins_D_vs_TreeSHAP_KGpart {'diff': 0.0511, 'D_better': 0.6875, 'ties': 0.07, 'p': 0.0}
   wil_del_D_vs_random {'diff': -0.1579, 'D_better': 0.865, 'ties': 0.0375, 'p': 0.0}
   wil_ins_D_vs_random {'diff': 0.1316, 'D_better': 0.8625, 'ties': 0.0425, 'p': 0.0}
===== hyb|xgb n 800 acc 0.936 k_med 5.0
  D                del 0.470  ins 0.936 pars 3.20 cov 0.991 cons 0.988 cont 0.944 t 0.0